In [ ]:
# Install Dependencies
!pip install -q langchain langchain-core langchain-openai

In [ ]:
# Set API Key
import os
from google.colab import userdata

try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except:
    pass

assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in Colab secrets'

In [ ]:
# Import Libraries
from typing import List, Optional
from dataclasses import dataclass, field

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [ ]:
# Build Travel Agent Chatbot

# Memory: Stores conversation messages in a Python list
@dataclass
class ConversationState:
    messages: List[BaseMessage] = field(default_factory=list)  # Python list for message history
    
    def add_message(self, role: str, content: str):
        # Append messages to the list (memory)
        if role == "human":
            self.messages.append(HumanMessage(content=content))
        elif role == "ai":
            self.messages.append(AIMessage(content=content))
    
    def get_messages_for_prompt(self) -> List[BaseMessage]:
        # Return copy of message list for LLM prompt context
        return list(self.messages)

class TravelAgentChatbot:
    # Domain filter: keywords to identify travel-related queries
    TRAVEL_KEYWORDS = [
        "flight", "flights", "hotel", "hotels", "destination", "destinations",
        "travel", "trip", "vacation", "booking", "reservation", "airport",
        "airline", "ticket", "tour", "sightseeing", "passport", "visa",
        "luggage", "baggage", "check-in", "boarding", "itinerary",
        "rental", "car", "taxi", "transport", "train", "bus", "cruise",
        "beach", "mountain", "city", "country", "restaurant", "food",
        "weather", "climate", "season", "budget", "cost", "price", "cheap",
        "expensive", "deal", "discount", "package", "all-inclusive", "tips"
    ]
    
    OUT_OF_SCOPE_RESPONSE = "I can't help with it."
    
    def __init__(self, llm=None):
        self.state = ConversationState()  # Initialize memory
        self.llm = llm
        self._setup_prompt()
    
    def _setup_prompt(self):
        # LangChain prompt template with system instruction and message history placeholder
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a Travel Agent. Only answer travel-related questions. For non-travel questions, reply exactly: I can't help with it."),
            MessagesPlaceholder(variable_name="history"),  # Injects message list here
            ("human", "{input}")
        ])
    
    def _is_travel_related(self, query: str) -> bool:
        # Check if query contains any travel keyword
        query_lower = query.lower()
        return any(kw in query_lower for kw in self.TRAVEL_KEYWORDS)
    
    def chat_stream(self, user_input: str):
        # Streaming: yields tokens one at a time for real-time output
        if not self.llm:
            raise ValueError("No LLM configured.")
        
        # Domain filter: reject non-travel queries
        if not self._is_travel_related(user_input):
            self.state.add_message("human", user_input)      # Store user message in memory
            self.state.add_message("ai", self.OUT_OF_SCOPE_RESPONSE)  # Store response in memory
            yield self.OUT_OF_SCOPE_RESPONSE
            return
        
        self.state.add_message("human", user_input)  # Store user message in memory
        
        # Build chain: prompt -> LLM -> string output
        chain = self.prompt | self.llm | StrOutputParser()
        
        # Stream tokens one by one (token-by-token output)
        full_response = ""
        for token in chain.stream({
            "history": self.state.get_messages_for_prompt(),  # Pass memory to LLM
            "input": user_input
        }):
            full_response += token
            yield token  # Yield each token for streaming display
        
        self.state.add_message("ai", full_response)  # Store complete response in memory
    
    def get_history(self) -> List[dict]:
        # Return conversation history from memory list
        return [{"role": type(m).__name__, "content": m.content} for m in self.state.messages]

In [ ]:
# Initialize Agent
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7)
agent = TravelAgentChatbot(llm=llm)

In [ ]:
# Test - Travel Query
query = 'What are the best destinations in Europe for summer?'
print(f"User: {query}")
print("Agent: ", end="")
for token in agent.chat_stream(query):
    print(token, end="")
print()

In [ ]:
# Test - Out of Scope Query
query = 'What is the capital of France?'
print(f"User: {query}")
print("Agent: ", end="")
for token in agent.chat_stream(query):
    print(token, end="")
print()

In [ ]:
# Interactive Chat
user_question = input("Ask the Travel Agent: ")
print(f"\nUser: {user_question}")
print("Agent: ", end="")
for token in agent.chat_stream(user_question):
    print(token, end="")
print()

In [ ]:
# View Conversation History
for msg in agent.get_history():
    print(f"[{msg['role']}] {msg['content'][:80]}...")